In [ ]:
!pip install --upgrade --no-index --find-links=/kaggle/input/transformers_package transformers -qq

In [ ]:
# qwen3-8b
!torchrun --nproc_per_node=2 inference.py \
  --model_name "/kaggle/input/qwen-3/transformers/8b/1" \
  --lora_path "/kaggle/input/qwen3-8b-models-adaptors" \
  --output_filename "submission_qwen3_8b_prob"

# qwen3-14b
!torchrun --nproc_per_node=2 inference.py \
    --model_name "/kaggle/input/qwen-3/transformers/14b/1" \
    --lora_path "/kaggle/input/qwen3-14b-models-adaptors" \
    --output_filename "submission_qwen3_14b_prob"

# qwen3-4b
!torchrun --nproc_per_node=2 inference.py \
    --model_name "/kaggle/input/qwen-3-4b-instruct-2507" \
    --lora_path "/kaggle/input/qwen3-4b-models-adaptors" \
    --output_filename "submission_qwen3_4b_prob"

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

# -------------------------
# 构建“family”映射：基于训练集中每个 QuestionId 下最常被“True_*”标记的 MC_Answer
# 用来为测试集每一行判定其“正确/错误”前缀（True_/False_）
# -------------------------
train = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/train.csv')  # 读训练集
test_df = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/test.csv')  # 读测试集

train['is_true'] = train['Category'].str.startswith('True')  # 布尔列：该条是否属于 True_* 类别

# 只保留最常见的正确选项作为“正确答案”
correct = (train[train.is_true]
           .assign(c=lambda df: df.groupby(['QuestionId','MC_Answer']).MC_Answer.transform('count'))
           .sort_values('c', ascending=False)
           .drop_duplicates(['QuestionId'])[['QuestionId','MC_Answer']])
correct['is_correct'] = 1  # 标记该 (QuestionId, MC_Answer) 为正确答案

# 最常见的正确选项作为“正确答案”，就 is_correct=1，否则填0
# 设置 row_id 为索引，映射为 {row_id: 'True_' 或 'False_'}
fam_map = (test_df.merge(correct, on=['QuestionId','MC_Answer'], how='left')
                  .assign(is_correct=lambda df: df.is_correct.fillna(0).astype(int))
                  .set_index('row_id')['is_correct']
                  .map({1: 'True_', 0: 'False_'}).to_dict())

# -------------------------
# 集成逻辑
# -------------------------
def extract_class_probabilities(row, model_suffix='', top_k=25):
    """从一行融合后的记录中抽取指定模型的类别与对应概率。
    
    参数:
        row: 合并后的一行记录（包含不同模型的列）
        model_suffix: 当前模型在合并后使用的列后缀（首个模型为空字符串）
        top_k: 最多读取的类别数（按 classes 列给出的顺序截断）
    返回:
        dict: {class_name: prob}，只包含当前模型可解析出的前 top_k 类别概率
    """
    classes_col = f'top_classes{model_suffix}'  # 当前模型的类别列表列名
    if classes_col in row:
        classes = row[classes_col].split(' ')[:top_k]  # 取前 top_k 个类别标签
    else:
        return {}  # 若不存在该列，返回空
    class_probs = {}
    for i in range(min(top_k, len(classes))):
        prob_col = f'prob_{i}{model_suffix}'  # 对应第 i 个类别的概率列名
        if prob_col in row:
            class_probs[classes[i]] = row[prob_col]  # 标签名: 概率值
    return class_probs


def ensemble_with_disagreement_handling(prob_files, model_weights=None, top_k=3):
    """对多个概率文件进行合并集成，并在类别前缀上应用“family”过滤。
    
    参数:
        prob_files: 各基模型输出的概率文件路径列表（CSV）；需含 row_id、top_classes、prob_i 列
        model_weights: 各模型的权重列表，长度应等于 n_models
        top_k: 从最终打分排序中选取的前 top_k 类别
    返回:
        list[str]: 与 merged 行对齐的预测字符串，每行以空格分隔多个 "Category:Misconception"
    """
    n_models = len(prob_files)  # 模型数量
    prob_dfs = []  # 存各文件 DataFrame
    final_predictions = []  # 存最终预测字符串

    # 读取每个模型的概率输出
    for file_path in prob_files:
        df = pd.read_csv(file_path)  
        prob_dfs.append(df)
    
    # 以 row_id 合并所有模型输出；第一个不加后缀，其余依次添加 _model{i+1} 后缀
    merged_df = prob_dfs[0]
    for i, df in enumerate(prob_dfs[1:], 1):
        merged_df = pd.merge(merged_df, df, on='row_id', suffixes=('', f'_model{i+1}'))

    # merged_df cols: row_id, top_classes, prob_0, ..., prob_24, top_classes_model2, prob_0_model2, ..., prob_24_model2, top_classes_model3, prob_0_model3, ..., prob_24_model3

    for idx, row in merged_df.iterrows():
        pref = fam_map[row['row_id']]  # 取该行的“family”前缀（'True_' 或 'False_'）
        
        # 抽取每个模型的类别-概率分布（最多取前25个，以覆盖更多候选）
        all_class_probs = []
        for i in range(n_models):
            suffix = f'_model{i+1}' if i > 0 else ''  # 除首个模型外都带后缀
            class_probs = extract_class_probabilities(row, suffix, top_k=25)
            all_class_probs.append(class_probs)
        
        # 汇总所有模型出现过的类别名称集合
        all_classes = set()
        for class_probs in all_class_probs:
            all_classes.update(class_probs.keys())
        
        # 统计“投票次数”“加权总概率”“加权最大概率”
        # 注意：此处假设 model_weights 长度与 n_models 一致
        class_votes = defaultdict(int)       # 记录该类别被多少模型命中（出现次数）
        class_total_prob = defaultdict(float)  # 记录该类别的加权概率总和
        class_max_prob = defaultdict(float)    # 记录该类别在各模型中的最大加权概率
        
        for i, class_probs in enumerate(all_class_probs):
            weight = model_weights[i]  # 当前模型权重
            for class_name, prob in class_probs.items():
                class_votes[class_name] += 1 # 投票次数
                class_total_prob[class_name] += prob * weight # 加权总概率
                class_max_prob[class_name] = max(class_max_prob[class_name], prob * weight) # 加权最大概率
        
        # 计算最终分数 = 加权总概率×0.34 + 发生一致性的占比×0.33 + 最大加权概率×0.33
        # 直觉上兼顾“总体支持度”“模型间一致性”“单模型置信峰值”
        final_scores = {}
        for class_name in all_classes:
            base_score = class_total_prob[class_name]
            agreement_bonus = class_votes[class_name] / n_models
            confidence_bonus = class_max_prob[class_name]
            final_scores[class_name] = (
                base_score * 0.34 +
                agreement_bonus * 0.33 +
                confidence_bonus * 0.33
            )
        
        # -------------------------
        # Family 过滤：仅保留与 pref ('True_'/'False_') 前缀一致的类别
        # -------------------------
        final_scores = {k: v for k, v in final_scores.items() if k.startswith(pref)}
        
        # 排序并取前 top_k 个类别
        sorted_classes = sorted(final_scores.items(), key=lambda x: -x[1])
        top_classes = [class_name for class_name, _ in sorted_classes[:top_k]]
        
        # 若候选不足3个，用“Neither:NA”优先补齐；在 True_ 情况下也再加一个 “Correct:NA”
        fillers = [f"{pref}Neither:NA"] + ([f"{pref}Correct:NA"] if pref == "True_" else [])
        for f in fillers:
            if len(top_classes) >= 3: 
                # 当已有3个及以上时，停止补齐
                break
            if f not in top_classes:
                # 若尚未包含该填充类别，则添加
                top_classes.append(f)

        while len(top_classes) < 3:
            top_classes.append(fillers[0])  # 最终保证至少3个
        
        final_predictions.append(' '.join(top_classes))  # 拼接为单行字符串
    
    return final_predictions


# -------------------------
# 执行集成
# -------------------------
weights = [
    1,1,1
]

prob_files = [
    '/kaggle/working/submission_qwen3_4b_prob.csv',
    '/kaggle/working/submission_qwen2_8b_prob.csv',
    '/kaggle/working/submission_qwen3_14b_prob.csv',
]

# 进行集成预测；此处 top_k=8 表示最终挑选前8个类别
predictions = ensemble_with_disagreement_handling(
    prob_files, 
    model_weights=weights,  
    top_k=8
)

# 组装提交文件，列名需与评测平台要求一致
submission = pd.DataFrame({
    'row_id': test_df.row_id.values,
    'Category:Misconception': predictions
})

submission.to_csv('submission.csv', index=False)  # 保存提交文件
print(submission.head())